## Import Library

In [ ]:
import os
import sys

# deteksi notebook run di colab / local
if 'google.colab' in sys.modules:
    
    # clone repo ke server temporary Colab
    !git clone https://github.com/stikkeju/tiket_pesawat.git
    
    # masuk ke folder notebook di dalam hasil kloning
    os.chdir('/content/tiket_pesawat/notebook')
else:
    print("Run di local env.")
    
print(f"Direktori aktif saat ini: {os.getcwd()}")

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import time

import warnings
warnings.filterwarnings('ignore')

## Load dataset yang sudah melalui proses preprocessing dan encoding:

In [2]:
pd.set_option('display.max_seq_items', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

df_model = pd.read_csv('../data/data_encoded.csv')

In [3]:
df_model.head()

,kelas,durasi_menit,transit,bagasi_kg,cabin_baggage_kg,meal,entertainment,usb,power,seat_pitch_inch,refundable,reschedulable,visa_required,wifi_status,selisih_hari,hari_penerbangan,jarak_km,is_transit,jumlah_fasilitas,harga_idr_log,bandara_asal_CGK,bandara_asal_DPS,bandara_asal_HAN,bandara_asal_HLP,bandara_asal_KNO,bandara_asal_KUL,bandara_asal_PDG,bandara_asal_PKU,bandara_asal_PLM,bandara_asal_PNK,bandara_asal_SGN,bandara_asal_SIN,bandara_asal_SUB,bandara_asal_SZB,bandara_asal_UPG,bandara_asal_XSP,bandara_tujuan_BPN,bandara_tujuan_BTH,bandara_tujuan_CGK,bandara_tujuan_DAD,bandara_tujuan_DPS,bandara_tujuan_HAN,bandara_tujuan_HLP,bandara_tujuan_HUI,bandara_tujuan_JHB,bandara_tujuan_KNO,bandara_tujuan_KUL,bandara_tujuan_LOP,bandara_tujuan_PDG,bandara_tujuan_PEN,bandara_tujuan_PGK,bandara_tujuan_PKU,bandara_tujuan_PLM,bandara_tujuan_PNK,bandara_tujuan_PQC,bandara_tujuan_PXU,bandara_tujuan_SGN,bandara_tujuan_SIN,bandara_tujuan_SUB,bandara_tujuan_SZB,bandara_tujuan_TJQ,bandara_tujuan_TKG,bandara_tujuan_UPG,bandara_tujuan_VII,bandara_tujuan_YIA,model_pesawat_Airbus A320,model_pesawat_Airbus A330,model_pesawat_Airbus A350,model_pesawat_Boeing 737,model_pesawat_Boeing 777,model_pesawat_Boeing 787,model_pesawat_Bombardier CR1000,model_pesawat_C909,model_pesawat_Embraer E195,model_pesawat_Unknown,seat_layout_1-2-2,seat_layout_2-2,seat_layout_2-2-2,seat_layout_2-3,seat_layout_2-3-2,seat_layout_2-4-2,seat_layout_3-3,seat_layout_3-3-3,seat_layout_3-4-3,seat_layout_Unknown,seat_type_ANGLE_FLAT_SEAT,seat_type_BELOW_AVERAGE_LEGROOM,seat_type_CRADLE_RECLINER,seat_type_FULL_FLAT_POD,seat_type_FULL_FLAT_SEAT,seat_type_PRIVATE_SUITE,seat_type_RECLINER_SEAT,seat_type_STANDARD_LEGROOM,seat_type_Unknown,maskapai_final_9G,maskapai_final_AK,maskapai_final_CI,maskapai_final_FY,maskapai_final_GA,maskapai_final_ID,maskapai_final_IN,maskapai_final_IP,maskapai_final_IU,maskapai_final_IW,maskapai_final_JT,maskapai_final_KL,maskapai_final_MH,maskapai_final_MU,maskapai_final_OD,maskapai_final_QG,maskapai_final_QH,maskapai_final_QZ,maskapai_final_SJ,maskapai_final_SQ,maskapai_final_TK,maskapai_final_TR,maskapai_final_VJ,maskapai_final_VN,maskapai_final_VU,waktu_berangkat_kategori_EARLY_MORNING,waktu_berangkat_kategori_EVENING,waktu_berangkat_kategori_MORNING,waktu_tiba_kategori_EARLY_MORNING,waktu_tiba_kategori_EVENING,waktu_tiba_kategori_MORNING
0,0,255,1,20,7,1,1,1,0,30.0,0,0,0,0,1,3,1387.763162,1,3,15.560720,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1
1,0,645,1,20,7,1,1,1,0,30.0,0,0,0,0,1,3,1387.763162,1,3,15.697278,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
2,0,1235,1,25,7,1,1,1,0,30.0,1,1,0,1,1,3,1387.763162,1,4,15.720763,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
3,0,1375,1,25,7,1,1,1,1,32.0,1,1,0,1,1,3,1387.763162,1,5,15.771527,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,390,1,25,7,1,1,0,0,31.0,0,0,0,0,1,3,1387.763162,1,2,16.211686,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0


## Data Splitting

Data splitting adalah langkah penting dalam workflow machine learning untuk memastikan bahwa model yang dibangun dapat digeneralisasikan dengan baik pada data yang belum pernah dilihat. Ini dapat menghindari bias evaluasi dan mengoptimalkan model dengan benar, dan memberikan estimasi kinerja yang lebih akurat. 

Dataset akan dibagi menjadi data train dan data test yang akan digunakan untuk melatih model.

In [4]:
# memisahkan fitur(X) dan target (y) pada dataset
X = df_model.drop(columns=['harga_idr_log'])
y = df_model['harga_idr_log']

# membagi dataset menjadi data latih dan data test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# menghitung panjang/jumlah data
print("Jumlah data: ",len(X))
# menghitung panjang/jumlah data pada x_train
print("Jumlah data train: ",len(X_train))
# menghitung panjang/jumlah data pada x_test
print("Jumlah data test: ",len(X_test))

Jumlah data:  18885
Jumlah data train:  15108
Jumlah data test:  3777


## Pemodelan

Pemodelan dilakukan dengan melatih data train terhadap beberapa algoritma regresi, seperti Ridge Regression, Random Forest, dan XGBoost.  
Setelah pelatihan model terhadap data train dan data test diakukan, akan dihitung nilai MAE, MSE, RMSE dan R2 untuk kedua hasil prediksi sebagai metrik evaluasi model untuk menentukan model yang akan digunakan dan melalui tahap hyperparameter tuning.

1. MAE (Mean Absolute Error)  
MAE adalah rata-rata dari kesalahan dengan nilai absolut antara nilai sebenarnya dan nilai prediksi.  
2. MSE (Mean Squared Error)  
MSE adalah nilai rata-rata dari kuadrat kesalahan antara nilai sebenarnya dan nilai prediksi.  
3. RMSE (Root Mean Squared Error)  
RMSE adalah akar kuadrat dari MSE. Ini mengembalikan kesalahan ke dalam satuan yang sama dengan data sehingga lebih mudah diinterpretasikan.  
4. R-squared (R2)  
R-squared adalah salah satu metrik yang digunakan untuk mengevaluasi seberapa baik model regresi linear menjelaskan variasi dalam data. R-squared memberikan ukuran proporsi variasi dalam variabel dependen (output) yang dapat dijelaskan oleh variabel independen (input) dalam model. 

### Pemodelan Ridge Regression

Ridge Regression adalah teknik regularisasi yang digunakan dalam regresi linear untuk mengatasi masalah multikolinearitas dan overfitting.  
Ridge regression menambahkan penalti berupa jumlah kuadrat dari koefisien regresi ke dalam fungsi loss. Penalti yang diterapkan membuat koefisien regresi menjadi lebih kecil (shrinkage), tetapi tidak pernah menyetel mereka menjadi nol. Ini berarti semua variabel tetap akan digunakan dalam pembangunan model, meskipun dengan koefisien yang lebih kecil.  

Ridge regression lebih cocok digunakan ketika semua variabel diharapkan memiliki pengaruh yang kecil tetapi signifikan dan tidak ingin menghilangkan variabel dari model.

In [5]:
# inisiasi dan melatih model dengan ridge regression
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train, y_train)

# prediksi model ridge terhadap data train dan data test
ridge_pred_train = ridge_model.predict(X_train)
ridge_pred_test = ridge_model.predict(X_test)

Hitung metrik evaluasi model Ridge terhadap data train dan test:

In [6]:
# hitung metrik evalausi
metrics_ridge = {
    'Model': 'Ridge Regression',
    'MAE Train': mean_absolute_error(y_train, ridge_pred_train),
    'MSE Train': mean_squared_error(y_train, ridge_pred_train),
    'RMSE Train': np.sqrt(mean_squared_error(y_train, ridge_pred_train)),
    'R2 Train': r2_score(y_train, ridge_pred_train),
    'MAE Test': mean_absolute_error(y_test, ridge_pred_test),
    'MSE Test': mean_squared_error(y_test, ridge_pred_test),
    'RMSE Test': np.sqrt(mean_squared_error(y_test, ridge_pred_test)),
    'R2 Test': r2_score(y_test, ridge_pred_test)
}

df_metrics_ridge = pd.DataFrame([metrics_ridge])

print(df_metrics_ridge.to_string(index=False))

           Model  MAE Train  MSE Train  RMSE Train  R2 Train  MAE Test  MSE Test  RMSE Test  R2 Test
Ridge Regression   0.216949   0.079692    0.282297  0.834609  0.218518  0.081384   0.285279 0.825194


### Pemodelan Decision Tree

Decision Tree adalah algoritma machine learning yang sering digunakan dalam tugas klasifikasi dan regresi. Struktur dari algoritma ini mirip dengan bentuk pohon dengan setiap cabang mewakili keputusan atau percabangan dari data berdasarkan fitur-fitur yang ada.  
Struktur dasar dari Decision Tree melibatkan tiga komponen utama, yaitu akar (root node), node (decision node), dan daun (leaf node). Root node mewakili seluruh dataset dan menjadi titik awal untuk pemisahan data. Node-node di sepanjang cabang pohon mewakili keputusan yang diambil berdasarkan fitur tertentu, sedangkan leaf node adalah hasil akhir dari proses klasifikasi atau regresi, seperti label kelas atau nilai numerik.

In [7]:
# inisiasi dan training model DT
decision_tree = DecisionTreeRegressor(random_state=42)
decision_tree.fit(X_train, y_train)

# prediksi model DT terhadap data train dan data test
dt_pred_train = decision_tree.predict(X_train)
dt_pred_test = decision_tree.predict(X_test)

In [8]:
# hitung metrik evaluasi
metrics_dt = {
    'Model': 'Decision Tree',
    'MAE Train': mean_absolute_error(y_train, dt_pred_train),
    'MSE Train': mean_squared_error(y_train, dt_pred_train),
    'RMSE Train': np.sqrt(mean_squared_error(y_train, dt_pred_train)),
    'R2 Train': r2_score(y_train, dt_pred_train),
    'MAE Test': mean_absolute_error(y_test, dt_pred_test),
    'MSE Test': mean_squared_error(y_test, dt_pred_test),
    'RMSE Test': np.sqrt(mean_squared_error(y_test, dt_pred_test)),
    'R2 Test': r2_score(y_test, dt_pred_test)
}

df_metrics_dt = pd.DataFrame([metrics_dt])

print(df_metrics_dt.to_string(index=False))

        Model  MAE Train  MSE Train  RMSE Train  R2 Train  MAE Test  MSE Test  RMSE Test  R2 Test
Decision Tree   0.008472   0.001875      0.0433  0.996109  0.075973  0.026737   0.163516  0.94257


### Pemodelan Random Forest

Random Forest adalah algoritma ensemble learning yang menggabungkan beberapa Decision Tree untuk meningkatkan akurasi prediksi dan mengurangi risiko overfitting. Setiap pohon dalam Random Forest dilatih menggunakan subset acak dari data pelatihan dan subset acak dari fitur yang tersedia. Hasil akhir prediksi ditentukan melalui voting (untuk klasifikasi) atau rata-rata (untuk regresi) dari hasil semua pohon dalam model.

In [9]:
# inisiasi dan training model RF
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# prediksi model RF terhadap data train dan data test
rf_pred_train = rf_model.predict(X_train)
rf_pred_test = rf_model.predict(X_test)

In [10]:
# hitung metrik evaluasi
metrics_rf = {
    'Model': 'Random Forest',
    'MAE Train': mean_absolute_error(y_train, rf_pred_train),
    'MSE Train': mean_squared_error(y_train, rf_pred_train),
    'RMSE Train': np.sqrt(mean_squared_error(y_train, rf_pred_train)),
    'R2 Train': r2_score(y_train, rf_pred_train),
    'MAE Test': mean_absolute_error(y_test, rf_pred_test),
    'MSE Test': mean_squared_error(y_test, rf_pred_test),
    'RMSE Test': np.sqrt(mean_squared_error(y_test, rf_pred_test)),
    'R2 Test': r2_score(y_test, rf_pred_test)
}

df_metrics_rf = pd.DataFrame([metrics_rf])

print(df_metrics_rf.to_string(index=False))

        Model  MAE Train  MSE Train  RMSE Train  R2 Train  MAE Test  MSE Test  RMSE Test  R2 Test
Random Forest    0.03067    0.00373     0.06107   0.99226   0.07445  0.017504   0.132302 0.962403


### Pemodelan XGBoost

XGBoost (Extreme Gradient Boosting) adalah sebuah algoritma machine learning yang menggunakan teknik ensemble learning, yang menggabungkan prediksi dari beberapa model untuk meningkatkan kinerja keseluruhan. Pendekatan ini berbasis pada teknik boosting, di mana model-model lemah (weak learners), biasanya berupa pohon keputusan (decision trees) yang sederhana, digabungkan menjadi model yang kuat (strong learner).

In [11]:
# inisiasi dan training model XGBoost
xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1)
xgb_model.fit(X_train, y_train)

# prediksi model XGBoost terhadap data train dan data test
xgb_pred_train = xgb_model.predict(X_train)
xgb_pred_test = xgb_model.predict(X_test)

In [12]:
# hitung metrik evaluasi
metrics_xgb = {
    'Model': 'XGBoost',
    'MAE Train': mean_absolute_error(y_train, xgb_pred_train),
    'MSE Train': mean_squared_error(y_train, xgb_pred_train),
    'RMSE Train': np.sqrt(mean_squared_error(y_train, xgb_pred_train)),
    'R2 Train': r2_score(y_train, xgb_pred_train),
    'MAE Test': mean_absolute_error(y_test, xgb_pred_test),
    'MSE Test': mean_squared_error(y_test, xgb_pred_test),
    'RMSE Test': np.sqrt(mean_squared_error(y_test, xgb_pred_test)),
    'R2 Test': r2_score(y_test, xgb_pred_test)
}

df_metrics_xgb = pd.DataFrame([metrics_xgb])

print(df_metrics_xgb.to_string(index=False))

  Model  MAE Train  MSE Train  RMSE Train  R2 Train  MAE Test  MSE Test  RMSE Test  R2 Test
XGBoost     0.1103   0.024252     0.15573  0.949668  0.116974  0.028657   0.169283 0.938448


### Kompilasi & Interpretasi Metrik Evaluasi Model Baseline

#### Kompilasi Metrik Evaluasi pada Model Baseline

In [13]:
# menggabungkan semua metrik evaluasi ke satu dataframe
df_comparison = pd.DataFrame([metrics_ridge, metrics_dt, metrics_rf, metrics_xgb])

# hitung gap R2 untuk mendeteksi overfitting
df_comparison['Gap R2'] = df_comparison['R2 Train'] - df_comparison['R2 Test']

kolom_urut = [
    'Model', 
    'MAE Train', 'MAE Test', 
    'MSE Train', 'MSE Test', 
    'RMSE Train', 'RMSE Test', 
    'R2 Train', 'R2 Test', 'Gap R2'
]
df_comparison = df_comparison[kolom_urut]

print("Tabel Kompilasi Metrik Evaluasi Model Baseline:")
display(df_comparison)

Tabel Kompilasi Metrik Evaluasi Model Baseline:


,Model,MAE Train,MAE Test,MSE Train,MSE Test,RMSE Train,RMSE Test,R2 Train,R2 Test,Gap R2
0,Ridge Regression,0.216949,0.218518,0.079692,0.081384,0.282297,0.285279,0.834609,0.825194,0.009415
1,Decision Tree,0.008472,0.075973,0.001875,0.026737,0.043300,0.163516,0.996109,0.942570,0.053538
2,Random Forest,0.030670,0.074450,0.003730,0.017504,0.061070,0.132302,0.992260,0.962403,0.029856
3,XGBoost,0.110300,0.116974,0.024252,0.028657,0.155730,0.169283,0.949668,0.938448,0.011221


#### Interpretasi Metrik Evaluasi terhadap model baseline:

##### Interpretasi model Rigde Regression:

Model Ridge Regression menghasilkan R2 Test (0.825194) dengan tingkat kesalahan tertinggi (MAE Test = 0.2185, RMSE Test = 0.2853 walaupun nilai Gap R2 (0.0091425) terkecil dibandingkan model lain. Nilai Gap R2 yang kecil menunjukkan bahwa model ini sangat stabil. Namun, karena performanya paling rendah di antara semua model, ini membuktikan adanya keterbatasan linieritas (underfitting relatif). Algoritma linear tidak mampu menangkap pola hubungan non-linear yang kompleks antara fitur-fitur tiket pesawat (seperti kombinasi rute, waktu, dan jenis maskapai) terhadap harga.

##### Interpretasi model Decision Tree:

Model Decision Tree menghasilkan R2 Train tertinggi (0.996109) namun menurun pada R2 Test (0.996109) dengan Gap R2 5.35% (0.053538) terbesar diantar model lainnya. Kemudian jika melihat nilai MAE Test (0.075973) mengalami lonjakan dibandingkan MAE Train (0.0084272). Ini adalah indikasi tekstual yang sangat jelas bahwa pohon keputusan tunggal tanpa batasan parameter mengalami overfitting parah.

##### Interpretasi model Random Forest:

Model Random Forest memiliki nilai MAE Test (0.074450) dan RMSE Test (0.132302) terendah dan nilai R2 Test tertinggi (0.962403) dibandingkan model lainnya. Meskipun nilai Gap R2 berada di angka 0.0299 (2.99%), angka ini masih berada dalam batas yang sangat aman dan wajar untuk sebuah model ensemble bawaan sebelum di-tuning. Sifat algoritma ini yang membangun ratusan pohon secara acak terbukti sangat superior dalam menjinakkan variansi data tiket penerbangan.

##### Interpretasi model XGBoost:

Model XGBoost menjadi model berbasis pohon yang paling seimbang dan konsisten dengan nilai Gap R2 hanya sebesar 0.0112 (1.12%). Namun, secara akurasi murni dan tingkat keperbawaan kesalahan (error), posisinya berada di bawah Random Forest pada dataset ini.

#### Keputusan Model Final:

Random Forest dipilih menjadi model final karena memberikan performa meminimalkan kesalahan prediksi pada data test (MAE dan RMSE terkecil daripada model lain). Masalah kesenjangan overfitting ringan sebesar 2.99% (nilai Gap R2) tersebut justru menjadi justifikasi yang sempurna untuk melakukan hyperparameter tuning pada tahap berikutnya, agar nilai gap dapat ditekan tanpa mengorbankan akurasinya yang sudah tinggi.

### Hyperparameter Tuning Model

Hyperparameter merupakan nilai yang tidak dipelajari selama pelatihan, tetapi dapat ditentukan sebelum pelatihan model dimulai. Proses hyperparameter tuning bertujuan untuk menemukan kombinasi hyperparameter yang menghasilkan kinerja model terbaik pada dataset tertentu. 

Hyperparameter yang terdapat pada model Random Forest:
1. 'n_estimators': parameter yang akan menentukan jumlah Decision Tree yang akan dibangun di dalam model Random Forest.
2. 'max_depth': mengatur kedalaman maksimum dari setiap Decision Tree dalam model Random Forest.
3. 'min_samples_split': jumlah minimum sampel yang diperlukan untuk membagi sebuah node dalam Decision Tree.
4. 'min_samples_leaf': jumlah minimum sampel yang diperlukan untuk berada pada leaf node (node daun).
5. 'max_features': jumlah maksimum fitur yang dipertimbangkan untuk pemisahan pada setiap node.
6. 'bootstrap': parameter yang menentukan bahwa sampel akan diambil dengan penggantian ketika membangun setiap Decision Tree.
7. 'random_state': parameter yang mengontrol pengacakan yang digunakan oleh algoritma.

Proses hyperparameter tuning akan dilakukan dengan Grid Search. Grid Search adalah salah satu metode hyperparameter tuning yang digunakan untuk menemukan kombinasi hyperparameter optimal pada model machine learning. Grid Search bekerja dengan mencoba semua kombinasi dari nilai hyperparameter yang telah kita tentukan dan mengevaluasi performa model untuk setiap kombinasi tersebut.

Tujuan dari Grid Search adalah untuk mengidentifikasi set hyperparameter yang menghasilkan performa terbaik berdasarkan metrik evaluasi yang dipilih (misalnya akurasi, F1-score, atau MSE).

#### Menentukan hyperparameter yang akan di-tuning:

In [14]:
param_grid_rf = {
    'n_estimators': [100, 150, 200],
    'max_depth': [15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
}

#### Melakukan GridSearchCV untuk menentukan parameter terbaik:

In [15]:
start_time = time.time()

# inisiasi GridSearchCV
rf_model_tune = RandomForestRegressor(random_state=42, n_jobs=1)
grid_search_rf = GridSearchCV(
    estimator=rf_model_tune,
    param_grid=param_grid_rf,
    cv=3,
    n_jobs=-1,
    verbose=1
)

# fir ke data train
grid_search_rf.fit(X_train, y_train)


best_params = grid_search_rf.best_params_

end_time = time.time()
execution_time = end_time - start_time

# output hasil terbaik
print(f"Parameter terbaik (Grid Search): {best_params}")
print(f"Waktu eksekusi: {execution_time:.4f} detik")

Fitting 3 folds for each of 81 candidates, totalling 243 fits
Parameter terbaik (Grid Search): {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}
Waktu eksekusi: 1212.0434 detik


#### Menggunakan parameter terbaik (best_params) dari GridSearchCV untuk melatih model Random Forest final:

In [16]:
# test best params untuk model rf_model_final

# inisiasi dan training model RF
rf_model_final = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
rf_model_final.fit(X_train, y_train)

# prediksi model RF terhadap data train dan data test
rf_final_pred_train = rf_model_final.predict(X_train)
rf_final_pred_test = rf_model_final.predict(X_test)

#### Menghitung metrik evaluasi (MAE, MSE, RMSE, R2) dari model Random Forest yang sudah dituning:

In [17]:
# hitung metrik evaluasi
metrics_rf_final = {
    'Model': 'Random Forest',
    'MAE Train': mean_absolute_error(y_train, rf_final_pred_train),
    'MSE Train': mean_squared_error(y_train, rf_final_pred_train),
    'RMSE Train': np.sqrt(mean_squared_error(y_train, rf_final_pred_train)),
    'R2 Train': r2_score(y_train, rf_final_pred_train),
    'MAE Test': mean_absolute_error(y_test, rf_final_pred_test),
    'MSE Test': mean_squared_error(y_test, rf_final_pred_test),
    'RMSE Test': np.sqrt(mean_squared_error(y_test, rf_final_pred_test)),
    'R2 Test': r2_score(y_test, rf_final_pred_test),
}

df_metrics_rf_final = pd.DataFrame([metrics_rf_final])

print(df_metrics_rf_final.to_string(index=False))

        Model  MAE Train  MSE Train  RMSE Train  R2 Train  MAE Test  MSE Test  RMSE Test  R2 Test
Random Forest   0.043195   0.005898    0.076799  0.987759  0.077464  0.017732   0.133162 0.961913


Nilai MAE Train pada 'rf_model_final' meningkat menjadi 0.0432 (4.32%) daripada baseline 0.0307 (3.07%). karena parameter untuk 'min_samples_split' dari hasil tuning dinaikkan menjadi 5, agar pohon pada Random Forest dipangkas tidak terlalu menghafal data train sehingga menghasilkan generalisasi lebih stabil.  
Nilai Gap R2 pada 'rf_model_final' menurun jadi 0.0258 (2.58%) dibandingkan dengan model baseline yang punya gap 2.99%, menunjukkan model berhasil memangkas jarak overfitting.

#### Skor importance fitur pada model Random Forest yang sudah dituning:

In [18]:
# Ambil skor kepentingan fitur dari model hasil tuning terbaikmu
importances = rf_model_final.feature_importances_
feature_names = X.columns

# Buat DataFrame untuk visualisasi
df_importance = pd.DataFrame({'Fitur': feature_names, 'Importance': importances})
df_importance = df_importance.sort_values(by='Importance', ascending=False).head(20)

print(df_importance)

                  Fitur  Importance
18     jumlah_fasilitas    0.312730
3             bagasi_kg    0.189276
1          durasi_menit    0.171509
8                 power    0.050824
16             jarak_km    0.043627
9       seat_pitch_inch    0.028779
56   bandara_tujuan_SIN    0.026610
7                   usb    0.022617
4      cabin_baggage_kg    0.013004
14         selisih_hari    0.011483
116   maskapai_final_VN    0.010523
10           refundable    0.009928
106   maskapai_final_MU    0.008712
30     bandara_asal_SIN    0.008352
15     hari_penerbangan    0.007720
37   bandara_tujuan_CGK    0.004652
112   maskapai_final_SQ    0.004576
0                 kelas    0.004511
2               transit    0.004085
107   maskapai_final_OD    0.004047


Interpretasi:
1. 'jumlah_fasilitas' menjadi faktor penentu paling dominan (31.27%).  
Hal ini sangat logis karena jumlah fasilitas (seperti makanan, hiburan di pesawat, selimut, dll.) berbanding lurus dengan jenis layanan maskapai (LCC vs. Full Service).

2. 'bagasi_kg' (18.92%) memiliki pengaruh besar, menunjukkan di industri penerbangan modern, kapasitas bagasi merupakan komponen biaya utama. Maskapai bertarif rendah (LCC) biasanya memangkas komponen ini untuk menekan harga baseline, sementara maskapai Full Service memasukkannya ke dalam komponen harga tiket regular, yang divalidasi dengan sangat baik oleh model.  

3. 'durasi_menit' (17.15%) lebih berpengaruh terhadap 'jarak_km' (4.36%) murni. Durasi penerbangan berkorelasi langsung dengan konsumsi bahan bakar per jam dan biaya operasional kru pesawat. Pesawat yang berputar-putar karena holding atau rute memutar memakan biaya lebih besar, sehingga durasi waktu menjadi prediktor biaya yang lebih sensitif dibanding jarak geografis antarkota.  

4. Fasilitas fisik seperti 'power' (5.08%), 'seat_pitch_inch' (2.87%) dan 'usb' (2.26%) menunjukkan kehadiran fasilitas fisik mempertegas klasifikasi tersembunyi antara armada pesawat modern/premium dengan armada pesawat ekonomi standar.

#### Export model Random Forest final ke .pkl:

In [19]:
joblib.dump(rf_model_final, '../artifact/rf_model_final.pkl', compress=3)
model_columns = list(X.columns)
joblib.dump(model_columns, '../artifact/model_columns.pkl')

['../artifact/model_columns.pkl']